In [14]:
import findspark
findspark.init()  # Initialize Spark
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



# Start Spark Sessie
spark = SparkSession.builder \
    .appName("CPI_Prediction_GBM") \
    .getOrCreate()

In [10]:
# Kolommen: CPI_prev, FED_rate, Unemployment, House_price, Oil_price, CPI_target (volgende maand)
df = pd.read_excel('BigBoy.xlsx')
# # 2. Zoek alle datum-kolommen en zet ze om naar strings (ISO formaat: YYYY-MM-DD)
# for col in pandas_df.columns:
#     if pandas_df[col].dtype == 'datetime64[ns]':
#         pandas_df[col] = pandas_df[col].dt.strftime('%Y-%m-%d')
# df = spark.createDataFrame(pandas_df)

In [11]:
display(df.head(5))

,Date,CPI,Avg Predicted CPI,Market Confidence Score,Trading Intensity,Fed Interest Rate (%),Unemployment Rate (%),Oil Prices (USD/bbl),Housing Prices,S&P 500,CPI After One Month
0,2026-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6343.72,NaN
1,2026-02-01,327.460,0.000,0.00,0.00,3.64,4.4,64.51,NaN,6878.88,NaN
2,2026-01-01,326.588,0.000,0.00,0.00,3.64,4.3,60.04,NaN,6939.03,327.460
3,2025-12-01,326.031,0.000,0.00,0.00,3.72,4.4,57.97,NaN,6845.50,326.588
4,2025-11-01,325.063,2.925,80.32,94594.62,3.88,4.5,60.06,NaN,6849.09,326.031


In [19]:
# Haal dynamisch de kolomnamen op
all_columns = df.columns
target_col = all_columns[-1]  # Pakt de allerlaatste kolom
feature_cols = all_columns[:-1] # Pakt alle kolommen BEHALVE de laatste

# Verwijder eventuele datum-kolommen uit de features (als die er nog in staan)
# Want een GBM kan alleen met getallen rekenen
feature_cols = [c for c in feature_cols if "datum" not in c.lower() and "date" not in c.lower()]

print(f"Target kolom: {target_col}")
print(f"Feature kolommen: {feature_cols}")

def preprocess_data(df):
    # Definieer de input features
    feature_cols = ["CPI_prev", "FED_rate", "Unemployment", "House_price", "Oil_price"]
    
    # 1. Assembleer features in een vector
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    
    # 2. Schalen van data (optioneel maar aanbevolen voor stabiliteit)
    scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
    
    return assembler, scaler

assembler, scaler = preprocess_data(df)

Target kolom: CPI After One Month
Feature kolommen: ['CPI', 'Avg Predicted CPI', 'Market Confidence Score', 'Trading Intensity', 'Fed Interest Rate (%)', 'Unemployment Rate (%)', 'Oil Prices (USD/bbl)', 'Housing Prices', 'S&P 500']


In [21]:
# Haal dynamisch de kolomnamen op
all_columns = df.columns.tolist()
target_col = all_columns[-1]  # Pakt de allerlaatste kolom
feature_cols = all_columns[1:-1] # Pakt alle kolommen BEHALVE de laatste

# Split the data (80% train, 20% test)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [22]:
print(f"feature_cols: {feature_cols}")

feature_cols: ['CPI', 'Avg Predicted CPI', 'Market Confidence Score', 'Trading Intensity', 'Fed Interest Rate (%)', 'Unemployment Rate (%)', 'Oil Prices (USD/bbl)', 'Housing Prices', 'S&P 500']


In [23]:
# 2. The only "must-have" step: Assembly
# This just bundles your numbers into a list for the model
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

# 3. Define the model pointing directly to the assembled features
gbt = GBTRegressor(featuresCol="raw_features", labelCol=target_col, maxIter=10)

# 4. Build and Train the simplified Pipeline
pipeline = Pipeline(stages=[assembler, gbt])
model = pipeline.fit(train_data)

AttributeError: 'DataFrame' object has no attribute '_jdf'

In [25]:
from pyspark.sql import SparkSession

# 1. Start a Spark Session (if you haven't already)
spark = SparkSession.builder.appName("ML_App").getOrCreate()

# 2. Convert Pandas DF to Spark DF
spark_df = spark.createDataFrame(df) 

# 3. Split the data using Spark's randomSplit (not train_test_split from sklearn)
train_data, test_data = spark_df.randomSplit([0.8, 0.2], seed=42)

# 4. Now the pipeline will work!
pipeline = Pipeline(stages=[assembler, gbt])
model = pipeline.fit(train_data)

OverflowError: mktime argument out of range

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# No VectorAssembler needed for sklearn!
X_train = train_data[feature_cols]
y_train = train_data[target_col]

model = GradientBoostingRegressor(n_estimators=10)
model.fit(X_train, y_train)